In [2]:
import os
import re
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()          # open the notebook from D:\upi-failure-economics
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = DATA_DIR / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Data dir  :", DATA_DIR)
print("Output dir:", OUTPUT_DIR)

Data dir  : d:\upi-failure-economics\data
Output dir: d:\upi-failure-economics\data\processed


In [3]:
def parse_month_year(filename):
    """Extract 'YYYY-MM' from a filename like 'remitter-april-2025.csv'."""
    match = re.search(r'([a-z]{3,9})[_-]?(\d{2,4})', filename.lower())
    if not match:
        return None
    m_str, y_str = match.groups()
    if len(y_str) == 2:
        y_str = '20' + y_str
    dt = pd.to_datetime(f"{m_str} {y_str}", format='%b %Y', errors='coerce')
    if pd.isna(dt):
        dt = pd.to_datetime(f"{m_str} {y_str}", format='%B %Y', errors='coerce')
    return None if pd.isna(dt) else dt.strftime('%Y-%m')


def clean_pct(val, is_fraction):
    if pd.isna(val):
        return 0.0
    val_str = str(val).strip()
    has_pct = '%' in val_str
    try:
        num = float(val_str.replace('%', '').replace(',', ''))
    except ValueError:
        return 0.0
    if is_fraction and not has_pct and 0 <= num <= 1.0:
        num *= 100
    return round(num, 4)

In [4]:
def load_one_file(file):
    """Read and clean a single NPCI file. Returns (role_type, clean_df) or None."""
    file_path = DATA_DIR / file
    file_lower = file.lower()

    # Role type
    if 'remitter' in file_lower:
        role_type, is_fraction = 'Remitter', True
    elif 'beneficiary' in file_lower:
        role_type, is_fraction = 'Beneficiary', False
    else:
        print(f"Skipping (not remitter/beneficiary): {file}")
        return None

    # Month-year
    month_year = parse_month_year(file)
    if month_year is None:
        print(f"Could not parse date from filename: {file}")
        return None

    # Read (skip subtitle row)
    try:
        if file_lower.endswith(('.xlsx', '.xls')):
            df = pd.read_excel(file_path, header=1)
        else:
            df = pd.read_csv(file_path, header=1)
    except Exception as e:
        print(f"Error reading {file}: {e}")
        return None

    # Standardize column names
    df.columns = (df.columns.str.strip().str.lower()
                  .str.replace(' ', '_').str.replace('%', 'pct'))

    # Identify columns
    try:
        bank_col = [c for c in df.columns if 'bank' in c or 'remitter' in c or 'beneficiary' in c][0]
        vol_col  = [c for c in df.columns if 'total_volume' in c or 'volume' in c][0]
        app_col  = [c for c in df.columns if 'approved' in c and 'pct' in c][0]
        bd_col   = [c for c in df.columns if 'bd' in c][0]
        td_col   = [c for c in df.columns if 'td' in c][0]
    except IndexError:
        print(f"Column mapping failed for {file}. Columns: {list(df.columns)}")
        return None

    # Build clean dataframe
    clean_df = pd.DataFrame({
        'month_year': month_year,
        'bank_name': df[bank_col].astype(str).str.strip(),
        'role_type': role_type,
        'total_volume_mn': pd.to_numeric(df[vol_col].astype(str).str.replace(',', ''), errors='coerce'),
        'approved_pct': df[app_col].apply(lambda v: clean_pct(v, is_fraction)),
        'technical_decline_pct': df[td_col].apply(lambda v: clean_pct(v, is_fraction)),
        'business_decline_pct': df[bd_col].apply(lambda v: clean_pct(v, is_fraction)),
    })

    # Drop bad rows, keep top 50
    clean_df = clean_df.dropna(subset=['bank_name', 'total_volume_mn'])
    clean_df = clean_df[~clean_df['bank_name'].str.lower().isin(['nan', '', 'sr.no.'])]
    return role_type, clean_df.head(50)

In [5]:
files = [f for f in os.listdir(DATA_DIR)
         if (DATA_DIR / f).is_file() and f.endswith(('.csv', '.xlsx', '.xls'))]
print(f"Found {len(files)} files in {DATA_DIR}")

remitter_dfs, beneficiary_dfs = [], []

for file in files:
    result = load_one_file(file)
    if result is None:
        continue
    role_type, clean_df = result
    (remitter_dfs if role_type == 'Remitter' else beneficiary_dfs).append(clean_df)

print(f"Remitter files processed   : {len(remitter_dfs)}")
print(f"Beneficiary files processed: {len(beneficiary_dfs)}")

Found 63 files in d:\upi-failure-economics\data
Could not parse date from filename: beneficiary_master_30_files.csv
Skipping (not remitter/beneficiary): npci_upi_clean_data.csv
Could not parse date from filename: remitter_master_30_files.csv
Remitter files processed   : 30
Beneficiary files processed: 30


In [6]:
def add_derived_columns(df):
    df['approved_volume_mn'] = (df['total_volume_mn'] * df['approved_pct'] / 100).round(2)
    df['total_decline_pct'] = (df['technical_decline_pct'] + df['business_decline_pct']).round(4)
    return df

remitter_master = add_derived_columns(pd.concat(remitter_dfs, ignore_index=True))
beneficiary_master = add_derived_columns(pd.concat(beneficiary_dfs, ignore_index=True))

remitter_master.to_csv(OUTPUT_DIR / 'remitter_master_30_files.csv', index=False)
beneficiary_master.to_csv(OUTPUT_DIR / 'beneficiary_master_30_files.csv', index=False)
print(f"Remitter master   : {len(remitter_master)} rows")
print(f"Beneficiary master: {len(beneficiary_master)} rows")

final_columns = [
    'month_year', 'bank_name', 'role_type', 'total_volume_mn',
    'approved_volume_mn', 'approved_pct', 'technical_decline_pct',
    'business_decline_pct', 'total_decline_pct'
]
final_master = pd.concat([remitter_master, beneficiary_master], ignore_index=True)[final_columns]

final_path = OUTPUT_DIR / 'npci_upi_clean_data.csv'
final_master.to_csv(final_path, index=False)
print(f"\nSUCCESS! Saved {final_path}")
print(f"Rows: {len(final_master)}, Columns: {len(final_master.columns)}")

Remitter master   : 1500 rows
Beneficiary master: 1500 rows

SUCCESS! Saved d:\upi-failure-economics\data\processed\npci_upi_clean_data.csv
Rows: 3000, Columns: 9


In [7]:
print(final_master.head())
print("\nRows per role/month:")
print(final_master.groupby(['role_type', 'month_year']).size().unstack(0))
print("\nNulls:\n", final_master.isna().sum())
print("\nMonths covered:", final_master['month_year'].nunique())

  month_year             bank_name role_type  total_volume_mn  \
0    2024-04   State Bank Of India  Remitter          3581.27   
1    2024-04         HDFC BANK LTD  Remitter          1157.04   
2    2024-04        Bank of Baroda  Remitter           898.17   
3    2024-04   Union Bank of India  Remitter           837.24   
4    2024-04  Punjab National Bank  Remitter           741.90   

   approved_volume_mn  approved_pct  technical_decline_pct  \
0             3369.98         94.10                   0.89   
1             1108.68         95.82                   0.09   
2              811.86         90.39                   1.59   
3              779.72         93.13                   1.59   
4              689.67         92.96                   1.11   

   business_decline_pct  total_decline_pct  
0                  5.01               5.90  
1                  4.08               4.17  
2                  8.02               9.61  
3                  5.28               6.87  
4          